# 03 — Preprocessing & Feature Engineering
**Project:** AI-Powered Essay Evaluation & Writing Assistant — Review 1

This notebook covers Review 1 Steps 5-9:
1. Data cleaning (missing/empty/duplicate/invalid checks)
2. Text preprocessing (`original_text` vs `clean_text`)
3. Separating X (input) and y (target)
4. Handcrafted NLP feature extraction
5. TF-IDF representation (explained here; actually *fit* later in notebook 04, after the train/test split, to avoid data leakage)
6. Saving the processed dataset to `data/processed/`

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')

from data.load_data import load_raw_data, DEMOGRAPHIC_COLUMNS
from data.preprocess import check_data_quality, clean_dataset, add_clean_text_column
from features.text_features import build_feature_dataframe

import nltk
# One-time downloads if not already present locally (safe to re-run).
for pkg in ['punkt', 'punkt_tab']:
    try:
        nltk.data.find(f'tokenizers/{pkg}')
    except LookupError:
        nltk.download(pkg, quiet=True)

RAW_PATH = '../data/raw/ASAP2_train_sourcetexts.csv'
PROCESSED_PATH = '../data/processed/essays_processed.csv'

df = load_raw_data(RAW_PATH)
df.shape

(24728, 14)

## 1. Data cleaning checks (Review 1 — Step 5)

We check for missing essays/scores, empty essays, duplicates, invalid scores, and whitespace issues
**before** deciding what to do about them — nothing gets silently dropped.

In [2]:
quality_report = check_data_quality(df)
quality_report

,count
missing_essay_id,0
missing_full_text,0
missing_score,0
empty_or_whitespace_only_essay,0
duplicate_essay_id,0
duplicate_essay_text,0
invalid_score_non_numeric,0
leading_or_trailing_whitespace_in_essay,15058


**Decisions** (implemented in `clean_dataset()` in `src/data/preprocess.py`, explained plainly, not hidden in a black box):
- Rows missing `full_text` or `score` are dropped — neither the input nor the label can be
  reconstructed, so the row is unusable for supervised learning.
- Empty/whitespace-only essays are dropped for the same reason.
- Duplicate `essay_id` rows are dropped (kept first occurrence) so no single essay is counted twice
  during training/evaluation.
- Leading/trailing whitespace in essay text is stripped (this does not remove any row).

We do **not** touch the demographic columns here — they are excluded from modeling entirely, not
"cleaned" as if they were a modeling input.

In [3]:
df_clean = clean_dataset(df)
df_clean.shape

[clean_dataset] 24728 -> 24728 rows (0 rows removed: missing/empty essay or score, or duplicate essay_id).


(24728, 14)

In [4]:
# Re-run the same checks after cleaning, to confirm the issues are resolved.
check_data_quality(df_clean)

,count
missing_essay_id,0
missing_full_text,0
missing_score,0
empty_or_whitespace_only_essay,0
duplicate_essay_id,0
duplicate_essay_text,0
invalid_score_non_numeric,0
leading_or_trailing_whitespace_in_essay,0


## 2. Text preprocessing — `original_text` vs `clean_text` (Review 1 — Step 5 continued)

We deliberately do **not** aggressively strip punctuation or stopwords here. Essay-quality signals
(sentence boundaries, punctuation use, capitalization patterns) matter for grammar/readability
analysis later, so destroying them at this stage would throw away useful information.

`clean_text()` only lowercases and collapses whitespace — nothing more. The untouched essay is kept
in `original_text` so nothing is lost.

In [5]:
df_clean = add_clean_text_column(df_clean)
df_clean[['essay_id', 'original_text', 'clean_text']].head(2)

,essay_id,original_text,clean_text
0,AAAVUP14319000159574,The author suggests that studying Venus is wor...,the author suggests that studying venus is wor...
1,AAAVUP14319000159542,NASA is fighting to be alble to to go to Venus...,nasa is fighting to be alble to to go to venus...


## 3. Separate input (X) and output (y) — Review 1 Step 6

For this baseline:
- `y` = `score` (the target)
- `X` = the essay text (`clean_text` / `original_text`) plus, shortly, the handcrafted numeric
  features below. `assignment`, `prompt_name`, and `source_text_*` are kept in the dataframe as
  optional contextual columns but are not forced into the baseline feature set — see the project
  README for why (Review 1 focuses on establishing a baseline from the essay text itself first).

In [6]:
y = df_clean['score']
X_text = df_clean['clean_text']

print('X_text shape:', X_text.shape)
print('y shape:', y.shape)

X_text shape: (24728,)
y shape: (24728,)


## 4. Handcrafted NLP features (Review 1 — Step 6)

Computed with NLTK (tokenization/sentence-splitting) and `textstat` (readability). This can take a
little while on the full ~24k-row dataset — that's expected, it's doing real NLP processing per
essay, not a lookup.

In [7]:
feature_df = build_feature_dataframe(df_clean['original_text'])
feature_df.head()

,word_count,character_count,sentence_count,avg_word_length,avg_sentence_length,sentence_length_std,unique_word_count,type_token_ratio,flesch_reading_ease,flesch_kincaid_grade
0,387,2329,17,4.809,22.765,12.129,200,0.5168,49.41,12.10
1,192,1019,10,4.115,19.200,12.057,113,0.5885,75.66,6.70
2,370,2141,31,4.605,11.935,3.951,186,0.5027,69.50,6.54
3,213,1251,10,4.601,21.300,10.060,112,0.5258,57.31,10.76
4,214,1230,7,4.593,30.571,20.472,127,0.5935,49.01,14.09


In [8]:
feature_df.describe()

,word_count,character_count,sentence_count,avg_word_length,avg_sentence_length,sentence_length_std,unique_word_count,type_token_ratio,flesch_reading_ease,flesch_kincaid_grade
count,24728.000000,24728.000000,24728.000000,24728.000000,24728.000000,24728.000000,24728.000000,24728.000000,24728.000000,24728.000000
mean,358.984956,2018.468416,18.686671,4.401431,21.967901,9.590944,162.184164,0.472334,63.409898,9.532287
std,146.923713,854.597496,8.612386,0.299017,17.366136,6.986445,50.944879,0.075544,15.888644,5.274200
min,147.000000,696.000000,1.000000,3.303000,6.331000,0.000000,33.000000,0.053700,-632.720000,-0.240000
25%,246.000000,1367.000000,12.000000,4.192750,16.125000,6.420000,124.000000,0.418300,56.647500,7.470000
50%,334.000000,1870.000000,18.000000,4.399000,19.105000,8.163500,155.000000,0.469100,64.400000,9.000000
75%,441.000000,2487.000000,24.000000,4.611000,23.176000,10.685500,192.000000,0.523200,71.730000,10.730000
max,1656.000000,8072.000000,133.000000,6.650000,714.000000,491.000000,462.000000,0.762400,108.520000,279.140000


**What each feature captures:**
- `word_count`, `character_count`, `sentence_count` — raw essay length.
- `avg_word_length`, `avg_sentence_length` — writing complexity at the word/sentence level.
- `sentence_length_std` — sentence length *variation*; very uniform sentence lengths can indicate
  simpler writing, high variation can indicate more sophisticated sentence structure (or run-ons).
- `unique_word_count`, `type_token_ratio` — vocabulary richness; type-token ratio is
  unique words / total words (naturally decreases as essays get longer, worth remembering when
  comparing essays of very different lengths).
- `flesch_reading_ease`, `flesch_kincaid_grade` — standard readability formulas.

None of these are ground-truth labels — they are *derived* signals, used as model input features,
not supervised targets (see Section 15 of the project brief / README).

In [9]:
df_features = pd.concat([df_clean.reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)
df_features.shape

(24728, 26)

## 5. TF-IDF representation — explained (Review 1 — Step 9)

```
Essay text
    |
Tokenization
    |
TF-IDF
    |
Numerical feature vectors
    |
ML model
```

`TfidfVectorizer` turns each essay into a sparse numeric vector where each dimension is a word (or
word pair) and the value reflects how important that term is to that essay relative to the whole
corpus. We'll use:
- unigrams + bigrams (`ngram_range=(1, 2)`) to capture some short phrases, not just single words
- `max_features` capped (e.g. 5000) so the matrix stays a manageable size
- `min_df` > 1 to drop extremely rare terms (typos, one-off words) that don't generalize

**Important — this is only a demonstration cell below.** We deliberately do **not** fit the real
TF-IDF vectorizer here on the full dataset. Fitting it now, before the train/test split, would leak
information from the test set into the vocabulary/IDF weights (the model would implicitly "see"
words from essays it's supposed to be evaluated on). The actual fit — `fit` on training text only,
`transform` on both train and test — happens in `04_baseline_models.ipynb`, right after the split.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

demo_vectorizer = TfidfVectorizer(max_features=20, ngram_range=(1, 2), min_df=2)
demo_matrix = demo_vectorizer.fit_transform(df_features['clean_text'].head(50))
print('Demo matrix shape (50 essays, capped at 20 terms):', demo_matrix.shape)
print('Example terms learned:', demo_vectorizer.get_feature_names_out()[:10])

Demo matrix shape (50 essays, capped at 20 terms): (50, 20)
Example terms learned: ['and' 'are' 'author' 'be' 'can' 'earth' 'in' 'is' 'it' 'of']


## 6. Save processed dataset

We save the cleaned essays + handcrafted features (not the TF-IDF matrix — that's regenerated
per-split in notebook 04 to keep training/test separation correct) to `data/processed/`.

In [11]:
cols_to_save = (
    ['essay_id', 'score', 'original_text', 'clean_text', 'assignment', 'prompt_name']
    + list(feature_df.columns)
)
df_to_save = df_features[cols_to_save]
df_to_save.to_csv(PROCESSED_PATH, index=False)
print(f'Saved {len(df_to_save)} rows x {len(df_to_save.columns)} columns to {PROCESSED_PATH}')

Saved 24728 rows x 16 columns to ../data/processed/essays_processed.csv


## 7. Summary

- Cleaning removed only rows with no usable essay/score or exact duplicate essays — every removal
  was counted and justified, nothing was silently dropped.
- `original_text` is preserved untouched; `clean_text` is only lowercased/whitespace-normalized, so
  punctuation and sentence structure (needed for grammar/readability analysis) are not destroyed.
- Handcrafted features (length, vocabulary richness, readability, sentence structure) were extracted
  per essay using NLTK + textstat.
- TF-IDF is explained and demonstrated on a small subset here, but the real fit happens strictly
  *after* the train/test split (Step 7, in notebook 04) to avoid data leakage.
- The processed dataset (cleaned text + handcrafted features) is saved to
  `data/processed/essays_processed.csv`, ready for `04_baseline_models.ipynb`.